<div style="border:1px solid #d0d0d0; padding:25px; border-radius:8px; background:#f8f9fa">

<h1 style="margin-bottom:10px">
Índices espectrales y segmentación con Landsat 9  
NDVI, umbralización Otsu, criterio operativo y NDBI
</h1>

<p>Material docente para la asignatura <b>Teledetección</b>.</p>

<b>Autor</b><br>
Julián Garzón Barrero<br>
Docente<br>
Programa de Ingeniería Topográfica y Geomática<br>
Universidad del Quindío<br>
juliangarzonb@uniquindio.edu.co

<hr>

<b>Apoyo técnico</b><br>
Estructuración y revisión computacional asistida mediante herramientas de inteligencia artificial (ChatGPT, OpenAI).

<hr>

<b>Propósito</b>

<p>
Analizar una imagen Landsat 9 previamente corregida atmosféricamente mediante DOS1, construir composiciones espectrales y calcular índices normalizados para explorar la respuesta de la vegetación y de las superficies construidas. El cuaderno enfatiza la diferencia entre un <b>índice espectral continuo</b> y una <b>decisión de segmentación</b>, así como la necesidad de mantener separados los píxeles válidos, las clases temáticas y los valores sin información.
</p>

</div>


---


## 📘 Módulo 1 — Entorno de trabajo y conexión a Google Drive

**Objetivo**  
Preparar el entorno de Google Colab, instalar las dependencias requeridas y conectar Google Drive para acceder al raster generado en la práctica anterior.

**Dependencias principales**

- `rasterio`: lectura y metadatos del GeoTIFF;
- `numpy`: operaciones matriciales;
- `matplotlib`: visualización;
- `scikit-image`: cálculo del umbral de Otsu.

**Resultado esperado**  
El entorno queda preparado y Google Drive disponible en `/content/drive`.


In [ ]:
# ============================================================
# MÓDULO 1 — Entorno de trabajo y conexión a Google Drive
# ============================================================

!pip -q install rasterio scikit-image matplotlib

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import rasterio
from skimage.filters import threshold_otsu
from google.colab import drive

# Conexión a Google Drive
drive.mount('/content/drive')

print("Entorno preparado correctamente.")


---


## 📘 Módulo 2 — Parámetros de entrada, lectura y control de calidad

**Objetivo**  
Leer el stack de reflectancia DOS1 y verificar que su estructura sea consistente con las bandas ópticas B2–B7 de Landsat 9.

**Entrada**  
GeoTIFF multibanda generado en la práctica de corrección atmosférica DOS1.

**Orden esperado de bandas**

1. B2 — Blue  
2. B3 — Green  
3. B4 — Red  
4. B5 — NIR  
5. B6 — SWIR1  
6. B7 — SWIR2

Se verifica el número de bandas, el sistema de referencia, el valor `nodata` y el número de píxeles con reflectancia válida. Para este ejercicio se considera físicamente admisible el intervalo `[0, 1]`, coherente con el producto DOS1 preparado previamente.


In [ ]:
# ============================================================
# MÓDULO 2 — Parámetros de entrada, lectura y control de calidad
# ============================================================

# ------------------------------------------------------------
# 1) Ruta del archivo de entrada
# ------------------------------------------------------------
DATA_DIR = Path("/content/drive/MyDrive/Colab Notebooks/datos_indices_espectrales")
STACK_PATH = DATA_DIR / "L9_DOS1_reflectance.tif"

if not STACK_PATH.exists():
    raise FileNotFoundError(
        f"No se encontró el archivo:\n{STACK_PATH}\n"
        "Verifique la ruta en Google Drive."
    )

# ------------------------------------------------------------
# 2) Lectura del GeoTIFF
# ------------------------------------------------------------
with rasterio.open(STACK_PATH) as src:
    if src.count < 6:
        raise ValueError(
            f"El stack contiene {src.count} bandas; se esperaban al menos 6 (B2–B7)."
        )

    stack = src.read(indexes=[1, 2, 3, 4, 5, 6]).astype("float32")
    profile = src.profile.copy()
    crs = src.crs
    transform = src.transform
    nodata = src.nodata

# ------------------------------------------------------------
# 3) Correspondencia explícita de bandas
# ------------------------------------------------------------
B2, B3, B4, B5, B6, B7 = stack

band_names = [
    "B2 Blue",
    "B3 Green",
    "B4 Red",
    "B5 NIR",
    "B6 SWIR1",
    "B7 SWIR2",
]

# ------------------------------------------------------------
# 4) Función general de validez
# ------------------------------------------------------------
def valid_mask(*bands, physical_range=True):
    """Construye una máscara booleana común para las bandas suministradas."""
    mask = np.ones(bands[0].shape, dtype=bool)

    for band in bands:
        mask &= np.isfinite(band)

        if nodata is not None:
            mask &= (band != nodata)

        if physical_range:
            mask &= (band >= 0) & (band <= 1)

    return mask

valid_optical = valid_mask(B2, B3, B4, B5, B6, B7)

n_total = valid_optical.size
n_valid = int(valid_optical.sum())

print("Archivo           :", STACK_PATH.name)
print("Dimensiones stack :", stack.shape)
print("CRS               :", crs)
print("NoData            :", nodata)
print("Píxeles válidos   :", n_valid)
print(f"Cobertura válida   : {100 * n_valid / n_total:.2f} %")


---


## 📘 Módulo 3 — Inspección de bandas espectrales

**Objetivo**  
Examinar individualmente las seis bandas ópticas antes de construir composiciones o índices.

La visualización utiliza un estiramiento robusto entre los percentiles 2 y 98 de cada banda. Este procedimiento modifica únicamente la representación gráfica; **no altera los valores de reflectancia utilizados en los cálculos posteriores**.


In [ ]:
# ============================================================
# MÓDULO 3 — Visualización de bandas espectrales Landsat 9
# ============================================================

fig, axes = plt.subplots(2, 3, figsize=(16, 10), dpi=160)

for i, (band, name) in enumerate(zip(stack, band_names)):
    valid_band = valid_mask(band)

    if not np.any(valid_band):
        raise ValueError(f"La banda {name} no contiene píxeles válidos.")

    p2, p98 = np.percentile(band[valid_band], (2, 98))

    ax = axes.flat[i]
    im = ax.imshow(
        np.ma.masked_where(~valid_band, band),
        cmap="gray",
        vmin=p2,
        vmax=p98,
        interpolation="nearest",
    )

    ax.set_title(name, fontsize=12)
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.suptitle("Bandas espectrales Landsat 9 · reflectancia DOS1", fontsize=16)
plt.tight_layout()
plt.show()


---


## 📘 Módulo 4 — Composiciones espectrales Landsat 9

**Objetivo**  
Construir cuatro composiciones RGB para explorar visualmente diferencias espectrales entre vegetación, superficies urbanas, agua, suelo y zonas con distinta respuesta en NIR y SWIR.

**Composiciones utilizadas**

- **4-3-2 (Red–Green–Blue):** color natural;
- **5-4-3 (NIR–Red–Green):** falso color infrarrojo, útil para resaltar vegetación;
- **5-6-4 (NIR–SWIR1–Red):** enfatiza contrastes asociados con vegetación, humedad y superficies secas;
- **6-5-4 (SWIR1–NIR–Red):** facilita la separación visual entre vegetación, áreas construidas y otras superficies.

Los tonos observados deben interpretarse como **respuesta espectral**, no como evidencia directa de una propiedad biofísica. Por ejemplo, una respuesta brillante o coloreada sobre agua puede ser compatible con mayor turbidez o sedimentos en suspensión, pero esa interpretación requiere corroboración independiente.


In [ ]:
# ============================================================
# MÓDULO 4 — Composiciones espectrales Landsat 9
# ============================================================

# ------------------------------------------------------------
# Funciones auxiliares de visualización
# ------------------------------------------------------------
def stretch(band, mask, pmin=2, pmax=98):
    """Estiramiento robusto para visualización en el intervalo [0, 1]."""
    vals = band[mask]

    if vals.size == 0:
        raise ValueError("No hay valores válidos para calcular el estiramiento.")

    lo, hi = np.percentile(vals, (pmin, pmax))

    if hi <= lo:
        raise ValueError("El rango percentilar es degenerado; no puede aplicarse el stretch.")

    out = (band - lo) / (hi - lo)
    out = np.clip(out, 0, 1).astype("float32")
    out[~mask] = 0
    return out


def gamma_correction(img, gamma_value=0.85):
    """Corrección gamma aplicada únicamente a la visualización RGB."""
    return np.power(img, gamma_value)


# Máscara común para garantizar cobertura espacial comparable
valid_rgb = valid_mask(B2, B3, B4, B5, B6)

# ------------------------------------------------------------
# Composiciones
# ------------------------------------------------------------
rgb_432 = gamma_correction(np.dstack((
    stretch(B4, valid_rgb),
    stretch(B3, valid_rgb),
    stretch(B2, valid_rgb),
)))

rgb_543 = gamma_correction(np.dstack((
    stretch(B5, valid_rgb),
    stretch(B4, valid_rgb),
    stretch(B3, valid_rgb),
)))

rgb_564 = gamma_correction(np.dstack((
    stretch(B5, valid_rgb),
    stretch(B6, valid_rgb),
    stretch(B4, valid_rgb),
)))

rgb_654 = gamma_correction(np.dstack((
    stretch(B6, valid_rgb),
    stretch(B5, valid_rgb),
    stretch(B4, valid_rgb),
)))

# ------------------------------------------------------------
# Visualización
# ------------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(16, 14), dpi=180)

composites = [
    (rgb_432, "RGB 4-3-2 · color natural"),
    (rgb_543, "RGB 5-4-3 · falso color infrarrojo"),
    (rgb_564, "RGB 5-6-4 · NIR–SWIR1–Red"),
    (rgb_654, "RGB 6-5-4 · SWIR1–NIR–Red"),
]

for ax, (rgb, title) in zip(axes.flat, composites):
    ax.imshow(rgb, interpolation="nearest")
    ax.set_title(title)
    ax.axis("off")

plt.suptitle("Composiciones espectrales Landsat 9", fontsize=18)
plt.tight_layout()
plt.show()


---


## 📘 Módulo 5 — NDVI, distribución estadística y umbral de Otsu

**Objetivo**  
Calcular el índice de vegetación de diferencia normalizada (NDVI) a partir de Red (B4) y NIR (B5), analizar su distribución estadística y utilizar el método de Otsu como ejemplo de umbralización automática.

El NDVI se calcula como:

$$
NDVI = \frac{NIR - Red}{NIR + Red}
$$

El índice toma valores teóricos en `[-1, 1]`. En términos generales, valores positivos altos suelen asociarse con una respuesta espectral dominada por vegetación; valores próximos a cero pueden corresponder a suelo desnudo, superficies construidas o mezclas espectrales; y valores negativos son frecuentes sobre agua u otras superficies con baja respuesta en NIR.

El NDVI debe interpretarse como un **indicador espectral o proxy del verdor y de la presencia de vegetación fotosintéticamente activa**. No constituye una medición directa de actividad fotosintética, biomasa o vigor fisiológico.

### Otsu: qué hace y qué no hace

El método de **Otsu** busca un umbral que divida una distribución en dos grupos minimizando la variabilidad interna de los grupos y, de forma equivalente, maximizando su separación estadística.

Su comportamiento suele ser más favorable cuando el histograma presenta una estructura **aproximadamente bimodal**, es decir, cuando existen dos poblaciones relativamente diferenciadas. Si las distribuciones se solapan fuertemente, existe una sola moda dominante o aparecen varias poblaciones, Otsu seguirá produciendo un valor numérico, pero ese valor **no tiene por qué corresponder a una frontera temática adecuada**.

Por tanto:

- Otsu **no reconoce vegetación, ciudad, agua ni suelo**;
- Otsu identifica una separación estadística en los valores de la imagen;
- la interpretación temática del umbral corresponde al analista;
- un umbral automático debe evaluarse espacialmente y, cuando el objetivo sea cartografía temática, validarse con información independiente.


In [ ]:
# ============================================================
# MÓDULO 5 — NDVI + histograma + umbral Otsu
# ============================================================

# ------------------------------------------------------------
# 1) Máscara de validez específica para NDVI
# ------------------------------------------------------------
valid_ndvi = valid_mask(B4, B5)

den_ndvi = B5 + B4
valid_ndvi &= np.isfinite(den_ndvi) & (den_ndvi != 0)

# ------------------------------------------------------------
# 2) Cálculo de NDVI
# ------------------------------------------------------------
NDVI = np.full(B4.shape, np.nan, dtype="float32")
NDVI[valid_ndvi] = (
    (B5[valid_ndvi] - B4[valid_ndvi]) /
    den_ndvi[valid_ndvi]
)

ndvi_vals = NDVI[np.isfinite(NDVI)]

if ndvi_vals.size == 0:
    raise ValueError("No hay valores válidos de NDVI.")

# ------------------------------------------------------------
# 3) Umbral Otsu
# ------------------------------------------------------------
th_ndvi_otsu = threshold_otsu(ndvi_vals)

print("Píxeles válidos para NDVI:", ndvi_vals.size)
print(f"NDVI mínimo              : {np.nanmin(ndvi_vals):.4f}")
print(f"NDVI máximo              : {np.nanmax(ndvi_vals):.4f}")
print(f"NDVI medio               : {np.nanmean(ndvi_vals):.4f}")
print(f"Umbral Otsu              : {th_ndvi_otsu:.4f}")

# ------------------------------------------------------------
# 4) Visualización espacial y distribución
# ------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 6), dpi=180)

im = axes[0].imshow(
    np.ma.masked_invalid(NDVI),
    cmap="RdYlGn",
    vmin=-1,
    vmax=1,
    interpolation="nearest",
)
axes[0].set_title("NDVI continuo")
axes[0].axis("off")
plt.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04)

axes[1].hist(ndvi_vals, bins=120, edgecolor="none")
axes[1].axvline(
    th_ndvi_otsu,
    linestyle="--",
    linewidth=2,
    label=f"Otsu = {th_ndvi_otsu:.4f}",
)
axes[1].set_title("Distribución de NDVI")
axes[1].set_xlabel("Valor NDVI")
axes[1].set_ylabel("Frecuencia")
axes[1].set_xlim(-1, 1)
axes[1].legend()

plt.tight_layout()
plt.show()


---

## 📘 Módulo 6 — Segmentación del NDVI: umbral automático frente a criterio operativo

**Objetivo**  
Comparar dos decisiones de segmentación aplicadas al mismo NDVI:

1. **Umbral Otsu:** criterio determinado automáticamente por la distribución estadística de la escena;
2. **NDVI > 0.40:** criterio operativo más restrictivo utilizado con fines comparativos en esta práctica.

La intención no es establecer que uno de los dos métodos sea universalmente superior. Se busca mostrar que **obtener un umbral es diferente de demostrar que ese umbral separa correctamente dos clases temáticas**.

En ambos casos se mantienen tres estados explícitos:

- `1`: píxel que satisface el criterio;
- `0`: píxel válido que no satisface el criterio;
- `NaN`: píxel sin información o inválido.

Esta distinción es fundamental: **NoData no constituye una clase temática**.

El valor `0.40` no debe interpretarse como un umbral universal de vegetación. Su utilidad depende de la cobertura, la fenología, el sensor, el preprocesamiento y las condiciones de adquisición. La comparación visual permite valorar coherencia espacial, pero no sustituye una validación independiente.


In [ ]:
# ============================================================
# MÓDULO 6 — Comparación de segmentaciones NDVI
# ============================================================

# ------------------------------------------------------------
# 1) Máscara basada en Otsu
# ------------------------------------------------------------
veg_otsu = np.full(NDVI.shape, np.nan, dtype="float32")
veg_otsu[valid_ndvi] = (
    NDVI[valid_ndvi] > th_ndvi_otsu
).astype("float32")

# ------------------------------------------------------------
# 2) Máscara basada en un criterio operativo fijo
# ------------------------------------------------------------
NDVI_THRESHOLD = 0.40

veg_040 = np.full(NDVI.shape, np.nan, dtype="float32")
veg_040[valid_ndvi] = (
    NDVI[valid_ndvi] > NDVI_THRESHOLD
).astype("float32")

# NDVI conservado únicamente donde se satisface el criterio 0.40
NDVI_veg_040 = np.full(NDVI.shape, np.nan, dtype="float32")
NDVI_veg_040[veg_040 == 1] = NDVI[veg_040 == 1]

# ------------------------------------------------------------
# 3) Resumen cuantitativo relativo a píxeles válidos
# ------------------------------------------------------------
n_valid_ndvi = int(valid_ndvi.sum())
n_otsu = int(np.sum(veg_otsu == 1))
n_040 = int(np.sum(veg_040 == 1))

print(f"Umbral Otsu       : {th_ndvi_otsu:.4f}")
print(f"Umbral operativo  : {NDVI_THRESHOLD:.2f}")
print(f"Vegetación Otsu   : {n_otsu} píxeles ({100*n_otsu/n_valid_ndvi:.2f} % de los válidos)")
print(f"Vegetación > 0.40 : {n_040} píxeles ({100*n_040/n_valid_ndvi:.2f} % de los válidos)")

# ------------------------------------------------------------
# 4) Colormap: NoData queda explícitamente separado
# ------------------------------------------------------------
mask_cmap = plt.cm.Greens.copy()
mask_cmap.set_bad("lightgray")

# ------------------------------------------------------------
# 5) Visualización comparativa
# ------------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(16, 14), dpi=180)

axes[0, 0].imshow(rgb_543, interpolation="nearest")
axes[0, 0].set_title("RGB 5-4-3 · referencia visual")
axes[0, 0].axis("off")

im_ndvi = axes[0, 1].imshow(
    np.ma.masked_invalid(NDVI),
    cmap="RdYlGn",
    vmin=-1,
    vmax=1,
    interpolation="nearest",
)
axes[0, 1].set_title("NDVI continuo")
axes[0, 1].axis("off")
plt.colorbar(im_ndvi, ax=axes[0, 1], fraction=0.046, pad=0.04)

axes[1, 0].imshow(
    np.ma.masked_invalid(veg_otsu),
    cmap=mask_cmap,
    vmin=0,
    vmax=1,
    interpolation="nearest",
)
axes[1, 0].set_title(f"Segmentación Otsu · NDVI > {th_ndvi_otsu:.4f}")
axes[1, 0].axis("off")

axes[1, 1].imshow(
    np.ma.masked_invalid(veg_040),
    cmap=mask_cmap,
    vmin=0,
    vmax=1,
    interpolation="nearest",
)
axes[1, 1].set_title("Criterio operativo · NDVI > 0.40")
axes[1, 1].axis("off")

plt.suptitle("Comparación de criterios de segmentación del NDVI", fontsize=18)
plt.tight_layout()
plt.show()


### Interpretación de la comparación

Si el umbral Otsu resulta inferior a `0.40`, el criterio `NDVI > 0.40` será, por construcción, más restrictivo y seleccionará una proporción menor de la escena.

El punto central es que **ninguno de los dos umbrales se convierte automáticamente en una verdad temática**. Otsu responde a la forma estadística de la distribución; el umbral `0.40` responde a una decisión operativa.

La composición 5-4-3 puede utilizarse como referencia cualitativa para discutir la coherencia espacial de ambas segmentaciones, pero sin muestras independientes no puede afirmarse cuál posee mayor exactitud temática.


---

## 📘 Módulo 7 — NDBI: índice continuo y análisis de su distribución

**Objetivo**  
Calcular el *Normalized Difference Built-up Index* (NDBI) a partir de SWIR1 (B6) y NIR (B5):

$$
NDBI = \frac{SWIR1 - NIR}{SWIR1 + NIR}
$$

Valores positivos pueden estar asociados con superficies construidas o superficies secas cuya reflectancia en SWIR1 supera la de NIR. Sin embargo, **NDBI no constituye por sí solo un clasificador de áreas urbanas**: suelo desnudo, materiales secos y otras coberturas pueden producir respuestas similares.

En este módulo se calcula únicamente el índice continuo y se analiza su distribución. La segmentación se deja deliberadamente para el módulo siguiente, donde el estudiante debe evaluar si Otsu resulta adecuado y, si no, calibrar un umbral alternativo para esta escena.


In [ ]:
# ============================================================
# MÓDULO 7 — Cálculo, visualización e histograma de NDBI
# ============================================================

# ------------------------------------------------------------
# 1) Máscara de validez específica para NDBI
# ------------------------------------------------------------
valid_ndbi = valid_mask(B5, B6)

den_ndbi = B6 + B5
valid_ndbi &= np.isfinite(den_ndbi) & (den_ndbi != 0)

# ------------------------------------------------------------
# 2) Cálculo de NDBI
# ------------------------------------------------------------
NDBI = np.full(B5.shape, np.nan, dtype="float32")
NDBI[valid_ndbi] = (
    (B6[valid_ndbi] - B5[valid_ndbi]) /
    den_ndbi[valid_ndbi]
)

ndbi_vals = NDBI[np.isfinite(NDBI)]

if ndbi_vals.size == 0:
    raise ValueError("No hay valores válidos de NDBI.")

# ------------------------------------------------------------
# 3) Estadísticos básicos
# ------------------------------------------------------------
print("Píxeles válidos para NDBI:", ndbi_vals.size)
print(f"NDBI mínimo              : {np.nanmin(ndbi_vals):.4f}")
print(f"NDBI máximo              : {np.nanmax(ndbi_vals):.4f}")
print(f"NDBI medio               : {np.nanmean(ndbi_vals):.4f}")

# ------------------------------------------------------------
# 4) Visualización
# ------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 6), dpi=180)

im = axes[0].imshow(
    np.ma.masked_invalid(NDBI),
    cmap="RdYlBu_r",
    vmin=-1,
    vmax=1,
    interpolation="nearest",
)
axes[0].set_title("NDBI continuo")
axes[0].axis("off")
plt.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04)

axes[1].hist(ndbi_vals, bins=120, edgecolor="none")
axes[1].axvline(0, linestyle="--", linewidth=2, label="NDBI = 0")
axes[1].set_title("Distribución de NDBI")
axes[1].set_xlabel("Valor NDBI")
axes[1].set_ylabel("Frecuencia")
axes[1].set_xlim(-1, 1)
axes[1].legend()

plt.tight_layout()
plt.show()


---

## 📘 Módulo 8 — Ejercicio: Otsu y calibración de un umbral para superficies construidas

**Propósito del ejercicio**  
Aplicar al NDBI la misma lógica utilizada previamente con NDVI, pero ahora sin asumir que el umbral automático resolverá correctamente el problema temático.

El estudiante debe:

1. calcular el umbral Otsu sobre la distribución de NDBI;
2. observar si el histograma presenta una estructura aproximadamente bimodal;
3. aplicar el umbral Otsu y evaluar espacialmente el resultado;
4. contrastar la segmentación con las composiciones espectrales;
5. proponer un **umbral alternativo calibrado para esta escena** cuando Otsu no produzca una separación temática satisfactoria;
6. justificar el valor seleccionado.

### Idea central

Un umbral obtenido para esta escena **no debe considerarse universal**. Puede cambiar cuando cambian el área de estudio, la fecha, la humedad, el estado de la vegetación, los materiales urbanos, el preprocesamiento o la distribución de coberturas incluida en el análisis.

La calibración mediante inspección del histograma y comparación espacial es un procedimiento exploratorio. Cuando el objetivo sea producir cartografía temática con rigor, debe complementarse con datos de referencia y una validación explícita.


In [ ]:
# ============================================================
# MÓDULO 8A — Umbral automático de NDBI mediante Otsu
# ============================================================

# ------------------------------------------------------------
# 1) Cálculo del umbral
# ------------------------------------------------------------
th_ndbi_otsu = threshold_otsu(ndbi_vals)

print(f"Umbral Otsu para NDBI: {th_ndbi_otsu:.4f}")

# ------------------------------------------------------------
# 2) Visualización del umbral sobre el histograma
# ------------------------------------------------------------
plt.figure(figsize=(10, 6), dpi=180)

plt.hist(ndbi_vals, bins=150, edgecolor="none")

plt.axvline(
    th_ndbi_otsu,
    linestyle="--",
    linewidth=2,
    label=f"Otsu = {th_ndbi_otsu:.4f}",
)

plt.axvline(
    0,
    linestyle=":",
    linewidth=2,
    label="NDBI = 0",
)

plt.xlabel("NDBI")
plt.ylabel("Frecuencia")
plt.title("Distribución de NDBI y umbral Otsu")
plt.xlim(-1, 1)
plt.legend()

plt.show()


### Antes de segmentar, observe el histograma

Preguntas de análisis:

- ¿La distribución presenta dos modos claramente diferenciados?
- ¿Existe un valle evidente entre dos poblaciones?
- ¿Dónde se ubica el umbral de Otsu respecto a esas poblaciones?
- ¿Hay razones suficientes para interpretar esas dos poblaciones como “construido” y “no construido”?

**Que Otsu entregue un valor no significa que existan dos clases temáticas bien separadas.**


In [ ]:
# ============================================================
# MÓDULO 8B — Segmentación NDBI mediante Otsu
# ============================================================

built_otsu = np.full(NDBI.shape, np.nan, dtype="float32")
built_otsu[valid_ndbi] = (
    NDBI[valid_ndbi] > th_ndbi_otsu
).astype("float32")

built_cmap = plt.cm.Greys.copy()
built_cmap.set_bad("lightgray")

plt.figure(figsize=(10, 8), dpi=180)

plt.imshow(
    np.ma.masked_invalid(built_otsu),
    cmap=built_cmap,
    vmin=0,
    vmax=1,
    interpolation="nearest",
)

plt.title(f"Segmentación NDBI mediante Otsu · T = {th_ndbi_otsu:.4f}")
plt.axis("off")
plt.show()


### Evalúe el resultado espacial

Compare la segmentación con las composiciones `4-3-2`, `5-6-4` y especialmente `6-5-4`.

Preguntas:

- ¿Las superficies seleccionadas corresponden principalmente a zonas construidas?
- ¿Aparecen suelos desnudos o superficies secas clasificadas como construidas?
- ¿Se omiten sectores urbanos?
- ¿El patrón espacial obtenido es coherente con lo observado en las composiciones?

Si la respuesta no es satisfactoria, no debe “forzarse” la interpretación de Otsu. El siguiente paso consiste en explorar un umbral alternativo y justificarlo.


In [ ]:
# ============================================================
# MÓDULO 8C — Calibración exploratoria de un umbral de NDBI
# ============================================================

# MODIFIQUE este valor y evalúe el resultado.
# No existe un valor universal que garantice separar
# superficies construidas de no construidas en cualquier escena.
NDBI_THRESHOLD = 0.00

built_manual = np.full(NDBI.shape, np.nan, dtype="float32")
built_manual[valid_ndbi] = (
    NDBI[valid_ndbi] > NDBI_THRESHOLD
).astype("float32")

n_valid_ndbi = int(valid_ndbi.sum())
n_otsu_built = int(np.sum(built_otsu == 1))
n_manual_built = int(np.sum(built_manual == 1))

print(f"Umbral Otsu              : {th_ndbi_otsu:.4f}")
print(f"Umbral exploratorio      : {NDBI_THRESHOLD:.4f}")
print(
    f"Seleccionado por Otsu    : {n_otsu_built} píxeles "
    f"({100*n_otsu_built/n_valid_ndbi:.2f} % de los válidos)"
)
print(
    f"Seleccionado manualmente : {n_manual_built} píxeles "
    f"({100*n_manual_built/n_valid_ndbi:.2f} % de los válidos)"
)


In [ ]:
# ============================================================
# MÓDULO 8D — Comparación visual de los dos criterios
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 7), dpi=180)

axes[0].imshow(rgb_654, interpolation="nearest")
axes[0].set_title("RGB 6-5-4 · referencia espectral")
axes[0].axis("off")

axes[1].imshow(
    np.ma.masked_invalid(built_otsu),
    cmap=built_cmap,
    vmin=0,
    vmax=1,
    interpolation="nearest",
)
axes[1].set_title(f"Otsu · NDBI > {th_ndbi_otsu:.3f}")
axes[1].axis("off")

axes[2].imshow(
    np.ma.masked_invalid(built_manual),
    cmap=built_cmap,
    vmin=0,
    vmax=1,
    interpolation="nearest",
)
axes[2].set_title(f"Umbral calibrado · NDBI > {NDBI_THRESHOLD:.3f}")
axes[2].axis("off")

plt.suptitle("NDBI · comparación entre umbral automático y umbral calibrado", fontsize=17)
plt.tight_layout()
plt.show()


### Preguntas finales del ejercicio

1. ¿El histograma de NDBI presenta una distribución claramente bimodal?
2. ¿Qué umbral obtuvo mediante Otsu?
3. ¿La segmentación obtenida con Otsu representa satisfactoriamente las superficies construidas?
4. ¿Qué coberturas producen las principales confusiones?
5. ¿Qué umbral alternativo seleccionó y cómo lo justificó?
6. ¿Esperaría obtener exactamente el mismo umbral en otra escena Landsat del mismo lugar tomada en otra fecha? Explique.
7. ¿Qué información adicional necesitaría para convertir esta exploración en una clasificación temática validada?

El objetivo no es encontrar un “número correcto” universal, sino comprender que el umbral depende de la distribución espectral concreta del área y de las condiciones de adquisición.


---

## Síntesis de la práctica

El flujo desarrollado separa cuatro niveles que no deben confundirse:

1. **Reflectancia:** variable radiométrica de entrada, previamente corregida mediante DOS1;
2. **Índice espectral:** transformación continua de la reflectancia, como NDVI o NDBI;
3. **Umbral:** regla estadística u operativa utilizada para dividir el espacio de valores;
4. **Clase temática:** interpretación final que requiere evidencia y, cuando se busca cartografía rigurosa, validación independiente.

La práctica muestra que:

- Otsu puede ser especialmente útil cuando la distribución presenta dos poblaciones bien diferenciadas;
- fuera de una estructura aproximadamente bimodal, su umbral debe examinarse críticamente;
- un umbral fijo puede ser útil de forma operativa, pero no debe asumirse como universal;
- los valores adecuados para una escena pueden cambiar en otra fecha, otra región o bajo otras condiciones;
- el algoritmo calcula números; la interpretación temática y la validación siguen siendo responsabilidad del analista.
